# ทดสอบ SAM 3 นับดอกไม้ (เทียบกับ OWLv2)

**ก่อนรัน**
1. Runtime → Change runtime type → **T4 GPU**
2. ขอสิทธิ์ที่ https://huggingface.co/facebook/sam3 (กด Request access) ให้ได้รับอนุมัติก่อน
3. สร้าง token แบบ Read ที่ https://huggingface.co/settings/tokens แล้วใส่ใน Colab: แถบซ้ายรูปกุญแจ 🔑 → Add new secret ชื่อ **`HF_TOKEN`** → เปิด Notebook access
4. อัปโหลด `sam3_test.zip` (อยู่ในโฟลเดอร์ flower) ไปไว้ใน Google Drive ที่ **`MyDrive/flower_yolo/sam3_test.zip`**

แล้ว Runtime → **Run all** ผลรูปจะถูกเซฟไว้ที่ `MyDrive/flower_yolo/sam3_out/`


In [ ]:
!pip -q install -U transformers pillow-avif-plugin

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, zipfile
OUT = '/content/drive/MyDrive/flower_yolo'
zipfile.ZipFile(f'{OUT}/sam3_test.zip').extractall('/content/sam3')
os.makedirs(f'{OUT}/sam3_out', exist_ok=True)
print(len(os.listdir('/content/sam3/test_images')), 'test images')

In [ ]:
import torch, json, glob
import pillow_avif  # noqa: F401
from PIL import Image
from transformers import Sam3Processor, Sam3Model
tok = userdata.get('HF_TOKEN')
dev = 'cuda'
model = Sam3Model.from_pretrained('facebook/sam3', token=tok).to(dev).eval()
proc = Sam3Processor.from_pretrained('facebook/sam3', token=tok)

def detect(img, prompt='flower'):
    inputs = proc(images=img, text=prompt, return_tensors='pt').to(dev)
    with torch.inference_mode():
        out = model(**inputs)
    # threshold ต่ำไว้ก่อน แล้วค่อยกรองทีหลังเพื่อลองหลายค่า
    r = proc.post_process_instance_segmentation(out, threshold=0.1, mask_threshold=0.5,
                                                target_sizes=inputs.get('original_sizes').tolist())[0]
    return r['boxes'].cpu().tolist(), r['scores'].cpu().tolist(), r['masks'].cpu()

In [ ]:
gt = json.load(open('/content/sam3/gt.json', encoding='utf-8')); gt.pop('_note')
OWL = {"02.jpg": 2, "0592353cf39da04db74a1d5cbd4e23da.jpg": 12, "16f082f426cb6c62905cd37cb3fb765f.jpg": 3, "55c9f80078472e95012a5ddee2e1511c.jpg": 4, "60827.webp": 16, "8fc9b5b7ed77061c826fc8956245a739.jpg": 0, "IMG_2537.jpg": 4, "IMG_7750.png": 7, "IMG_7751.png": 7, "N1924789_80.jpg": 4, "Screenshot 2026-10-20 164407.png": 14, "a7562bc0589d14a376f81a8b768c74abe8454d62-500x750.avif": 1, "aabe490b4e286bfdecf0d460061101d7.jpg": 7, "b1316bef14fc00672e7e0adaf461c7ee.jpg": 6, "d76dd07f92a43bda83d8db27daba87df.jpg": 10, "e01a69e227b6477a0a0bcff775f626f6.jpg": 3, "eea9e22bebac018a1adf6e6a854f2472.jpg": 9, "fcbf7439e4bd4b27cb02049b3c951ede.png": 1, "images.jpg": 0, "images_1.jpg": 14, "images_2.jpg": 2, "images_3.jpg": 19, "images_4.jpg": 1}   # ผลของ flower_counter_v4.9_owl2 (ผิดรวม 23, ตรง 12/23)
raw = {}
for n in gt:
    img = Image.open(f'/content/sam3/test_images/{n}').convert('RGB')
    raw[n] = detect(img)
    print(n[:30], 'scores:', [round(s, 2) for s in raw[n][1]][:25])

In [ ]:
print('threshold  ผิดรวม  ตรง')
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    cnt = {n: sum(s >= thr for s in raw[n][1]) for n in gt}
    print(f'{thr:.1f}        {sum(abs(cnt[n]-g) for n, g in gt.items()):4d}   {sum(cnt[n]==g for n, g in gt.items())}/{len(gt)}')
print(f"OWLv2      {sum(abs(OWL[n]-g) for n, g in gt.items()):4d}   {sum(OWL[n]==g for n, g in gt.items())}/{len(gt)}")
print('(รูปเฉลย 0 = ของปลอม/การ์ตูน ซึ่ง OWLv2 มีด่าน CLIP ปัดตกให้ แต่ SAM 3 ล้วนๆ ไม่มี)')

In [ ]:
THR = 0.5   # ← เปลี่ยนเป็นค่าที่ผิดน้อยสุดจากตารางข้างบน
print('รูป                              เฉลย  SAM3  OWLv2')
for n, g in gt.items():
    c = sum(s >= THR for s in raw[n][1])
    print(f'{n[:32]:32s} {g:4d} {c:5d} {OWL[n]:6d}')

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as pa, numpy as np
def show(folder, names, thr, save):
    for n in names:
        img = Image.open(f'{folder}/{n}').convert('RGB')
        b, s, m = raw[n] if n in raw else detect(img)
        keep = [i for i, x in enumerate(s) if x >= thr]
        fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(img); ax.axis('off')
        ov = np.zeros((img.size[1], img.size[0], 4))
        for i in keep:
            ov[m[i].numpy() > 0] = [1, 0, 0, 0.35]
            x0, y0, x1, y1 = b[i]; ax.add_patch(pa.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, ec='red', lw=2))
        ax.imshow(ov); ax.set_title(f'{n[:30]}  SAM3={len(keep)}')
        fig.savefig(f'{OUT}/sam3_out/{save}_{os.path.splitext(n)[0]}.jpg', bbox_inches='tight', dpi=80); plt.show()
show('/content/sam3/test_images', list(gt), THR, 'test')
show('/content/sam3/demo', sorted(os.listdir('/content/sam3/demo')), THR, 'demo')